# 🎮 GTA 6 Viral Clipper — Google Colab Edition
Gere cortes verticais virais automáticos para Reels, Shorts e TikTok direto na nuvem do Google!

### ⚡ Benefícios no Google Colab:
- **GPU Grátis (NVIDIA T4):** Transcrição com Whisper até 10x mais rápida do que no computador local.
- **Internet Gigabit:** Downloads parciais ultrarrápidos direto dos servidores do YouTube/Twitch.
- **Zero consumo local:** Não esquenta seu PC, não usa memória nem espaço em disco local.
- **Google Drive Integrado:** Todos os vídeos são salvos diretamente na sua conta do Google Drive.

In [2]:
#@title 📁 1. Conectar ao Google Drive
#@markdown Clique no Play. Se abrir a janela de permissão, selecione sua conta Google e clique em Permitir.
import os
try:
    from google.colab import auth
    auth.authenticate_user()
except Exception as e:
    print('Aviso auth:', e)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('✓ Google Drive montado com sucesso!')
except Exception as e:
    print('Erro ao montar:', e)
    print('DICA: Se der erro de credencial, clique no ícone de Pasta (Arquivos) na barra lateral esquerda e depois no botão com o logo do Google Drive para montar graficamente!')

Aviso auth: Error: credential propagation was unsuccessful
Erro ao montar: Error: credential propagation was unsuccessful
DICA: Se der erro de credencial, clique no ícone de Pasta (Arquivos) na barra lateral esquerda e depois no botão com o logo do Google Drive para montar graficamente!


In [ ]:
#@title ⚙️ 2. Instalar Dependências & Atualizar Projeto
#@markdown Instala FFmpeg, fontes Montserrat, bibliotecas CUDA 12, detector facial YuNet e sincroniza o código.
%cd /content
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-montserrat unzip
!pip install -q -U --pre "yt-dlp[default]"
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12
!pip install -q faster-whisper opencv-python-headless numpy certifi static-ffmpeg deno

# Baixar modelo de deteccao de rosto YuNet para Split-Screen
!mkdir -p /root/clipper-gta6/modelos
!curl -fsSL -o /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx

!rm -rf /content/meu-gta6-clipper
!git clone -q --depth 1 https://github.com/papaigato/gta6-auto-clips.git /content/meu-gta6-clipper
%cd /content/meu-gta6-clipper
print('✓ Ambiente configurado e código v3 (2K + GPU T4 Turbo) clonado do GitHub!')

In [ ]:
#@title 🍪 2.1 (Opcional) Subir cookies do YouTube
#@markdown Se o YouTube bloquear o IP do Colab com pedido de login, suba seu arquivo cookies.txt aqui uma vez só (ele fica salvo no seu Google Drive para sempre):
import shutil, os
from pathlib import Path

drive_cookie = Path('/content/drive/MyDrive/cookies.txt')
if drive_cookie.exists():
    print('✓ Cookies já encontrados no seu Google Drive! Não precisa subir de novo.')
else:
    try:
        from google.colab import files
        print('Selecione o arquivo cookies.txt do seu computador:')
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith('.txt'):
                shutil.copyfile(fn, '/content/drive/MyDrive/cookies.txt')
                shutil.copyfile(fn, '/content/cookies.txt')
                print('✓ Cookies salvos no Google Drive com sucesso!')
    except Exception as e:
        print('Nenhum cookie enviado (tentando modo automático com cliente Android/Web):', e)

In [ ]:
#@title 🎬 3. Gerar Cortes da Live (2K + GPU T4 Turbo)
#@markdown Escolha o link da live, quantidade, resolução 2K, FPS e estilo:

LINK_DA_LIVE = "https://youtu.be/Q9ZqVAdtASE" #@param {type:"string"}
QUANTIDADE_DE_CORTES = 4 #@param {type:"integer"}
RESOLUCAO_SAIDA = "Full HD (1080x1920 - Mais Rapido)" #@param ["2K (1440x2560 - Ultra Nitido)", "Full HD (1080x1920 - Mais Rapido)"]
FPS_SAIDA = "60 fps (Super Fluido)" #@param ["60 fps (Super Fluido)", "30 fps (2x Mais Rapido)"]
RESOLUCAO_DOWNLOAD = "1080p60 (Full HD)" #@param ["1440p60 (Ultra-HD 2K - Maxima Qualidade)", "1080p60 (Full HD)", "720p60 (Mais rapido)"]
ESTILO_VISUAL = "Split-Screen 30/70 (TikTok Grota Clips: Webcam Topo + Jogo)" #@param ["Split-Screen 30/70 (TikTok Grota Clips: Webcam Topo + Jogo)", "Fundo Desfocado (Blur - Estilo Video 8)", "Tela Cheia (Preencher)"]
CORTES_EM_PARALELO = 2 #@param [1, 2, 3] {type:"raw"}
REFAZER_CORTES_EXISTENTES = True #@param {type:"boolean"}
PASTA_DESTINO_DRIVE = "/content/drive/MyDrive/GTA6_Cortes" #@param {type:"string"}

import os, glob, sys
# Garante detector de rosto baixado no Colab
!mkdir -p /root/clipper-gta6/modelos
![ ! -f /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx ] && curl -fsSL -o /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx

# Atualiza o codigo mais recente do GitHub
%cd /content/meu-gta6-clipper
!git pull -q

# Garante bibliotecas CUDA 12 (cuBLAS/cuDNN) no LD_LIBRARY_PATH para Whisper rodar na GPU T4
nv_libs = [p for base in sys.path for p in glob.glob(os.path.join(base, 'nvidia', '*', 'lib')) if os.path.isdir(p)]
if nv_libs:
    atual_ld = os.environ.get('LD_LIBRARY_PATH', '')
    os.environ['LD_LIBRARY_PATH'] = ':'.join(dict.fromkeys(nv_libs + ([atual_ld] if atual_ld else [])))

# Salva cache de lives (transcricao, escolhidos) no Google Drive e arquivos pesados temporarios no SSD local (/tmp)
CACHE_DRIVE = f"{PASTA_DESTINO_DRIVE}/.cache"
!mkdir -p "$CACHE_DRIVE/modelos" /tmp/clipper_gta6
![ -f /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx ] && cp -u /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx "$CACHE_DRIVE/modelos/"

if REFAZER_CORTES_EXISTENTES:
    # Remove pedacos antigos para forcar o download limpo com pre-roll anti-travamento
    !rm -rf /tmp/clipper_gta6/*
    !rm -f /root/clipper-gta6/trabalho/*/*/trecho.* /root/clipper-gta6/trabalho/*/*/*.webm /root/clipper-gta6/trabalho/*/*/*.mkv /root/clipper-gta6/trabalho/*/*/feito.txt
    !rm -f "$CACHE_DRIVE/trabalho"/*/*/trecho.* "$CACHE_DRIVE/trabalho"/*/*/*.webm "$CACHE_DRIVE/trabalho"/*/*/*.mkv "$CACHE_DRIVE/trabalho"/*/*/feito.txt

max_res = "1440" if "1440" in RESOLUCAO_DOWNLOAD else ("1080" if "1080" in RESOLUCAO_DOWNLOAD else "720")
out_res = "2k" if "2K" in RESOLUCAO_SAIDA else "1080p"
fps_val = "30" if "30" in FPS_SAIDA else "60"
os.environ["CLIPPER_MAX_RES"] = max_res
os.environ["CLIPPER_OUT_RES"] = out_res
os.environ["CLIPPER_FPS"] = fps_val
os.environ["CLIPPER_PARALELO"] = str(CORTES_EM_PARALELO)
os.environ["CLIPPER_CORTES_DIR"] = PASTA_DESTINO_DRIVE
os.environ["CLIPPER_GTA6_HOME"] = CACHE_DRIVE
os.environ["CLIPPER_TMP_DIR"] = "/tmp/clipper_gta6"
os.environ["CLIPPER_ESTILO"] = "blur" if "Blur" in ESTILO_VISUAL else ("preencher" if "Preencher" in ESTILO_VISUAL else "split")
os.environ["CLIPPER_FORCAR_REFAZER"] = "1" if REFAZER_CORTES_EXISTENTES else "0"
os.environ["CLIPPER_ESPERA"] = "86400"

print(f"🚀 Link da Live: {LINK_DA_LIVE}")
print(f"💎 Saída: {RESOLUCAO_SAIDA} @ {fps_val}fps | Download: {max_res}p (VP9/H264 GPU)")
print(f"⚡ GPU T4 Paralelo: {CORTES_EM_PARALELO} cortes simultâneos | Pre-roll anti-travamento: ATIVO")
print(f"🎨 Estilo: {os.environ['CLIPPER_ESTILO']}")
print(f"♻️ Refazer cortes antigos: {REFAZER_CORTES_EXISTENTES}")
print(f"📁 Destino: {PASTA_DESTINO_DRIVE}")

!python /content/meu-gta6-clipper/scripts/clip.py automatico "$LINK_DA_LIVE" --quantos $QUANTIDADE_DE_CORTES


In [ ]:
#@title 📋 4. Verificar Cortes Salvos no Google Drive
import glob, os
videos = glob.glob(f"{PASTA_DESTINO_DRIVE}/*/*.mp4")
print(f"🎬 Total de vídeos prontos no Google Drive: {len(videos)}")
for v in sorted(videos)[-15:]:
    mb = os.path.getsize(v) / (1024 * 1024)
    print(f"✓ {os.path.basename(v)} ({mb:.1f} MB)")